In [1]:

import pathlib
import pandas as pd
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import os
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import classification_report, f1_score
from detach_rocket.detach_classes import DetachEnsemble
import cebra
from cebra import CEBRA

import tempfile
from pathlib import Path
import gc
import pickle
from pathlib import Path

from sklearn.manifold import TSNE

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization,
    LSTM, Bidirectional, GlobalAveragePooling1D
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import tensorflow as tf
import polars as pl
from sklearn.metrics import confusion_matrix

import numpy as np
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix



import pickle


/Users/ashhadulislam/miniconda3/envs/py310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
def minirocket_results(train_embedding,y_train,val_embedding,y_val,plot=True,num_kernels=1000):

    X_train_rocket = train_embedding.transpose(0, 2, 1)
    X_val_rocket  = val_embedding.transpose(0, 2, 1)

    print(X_train_rocket.shape)
    minirocket_pipeline = make_pipeline(
        MiniRocketMultivariate(num_kernels=num_kernels),                  # time-series transform
        StandardScaler(with_mean=False),            # scale features
        RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))  # linear classifier
    )
    minirocket_pipeline.fit(X_train_rocket, y_train)
    y_val_pred = minirocket_pipeline.predict(X_val_rocket)
    

    print(classification_report(y_val, y_val_pred))
    cm = confusion_matrix(y_val, y_val_pred, labels=[0, 1])
    acc = accuracy_score(y_val, y_val_pred)
    prec = precision_score(y_val, y_val_pred, pos_label=1)   # precision for Target=1
    rec = recall_score(y_val, y_val_pred, pos_label=1)       # recall for Target=1

    if plot:

        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal (0)", "Target (1)"])
        disp.plot(cmap="Blues", values_format="d")
        plt.title("MiniRocket + Ridge Confusion Matrix (Validation)")
        plt.show()
        print(f"Accuracy : {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall   : {rec:.4f}")
        print("Macro F1:", f1_score(y_val, y_val_pred, average="macro"))
        print("Weighted F1:", f1_score(y_val, y_val_pred, average="weighted"))
        print("Confusion Matrix:\n", cm)

    return {
        "acc": acc,
        "precision": prec,
        "recall": rec,
        "macro_f1": f1_score(y_val, y_val_pred, average="macro"),
        "weighted_f1": f1_score(y_val, y_val_pred, average="weighted"),
        "cm": cm,
        #"y_val_pred": y_val_pred,
        #"model": minirocket_pipeline,
    }




In [3]:


TARGET_GESTURES = [
    'Above ear - pull hair',
    'Cheek - pinch skin',
    'Eyebrow - pull hair',
    'Eyelash - pull hair',
    'Forehead - pull hairline',
    'Forehead - scratch',
    'Neck - pinch skin',
    'Neck - scratch',
]

NON_TARGET_GESTURES = [
    'Write name on leg',
    'Wave hello',
    'Glasses on/off',
    'Text on phone',
    'Write name in air',
    'Feel around in tray and pull out an object',
    'Scratch knee/leg skin',
    'Pull air toward your face',
    'Drink from bottle/cup',
    'Pinch knee/leg skin',
]

ALL_GESTURES_SET = TARGET_GESTURES + NON_TARGET_GESTURES



gesture2idx = {g: i for i, g in enumerate(ALL_GESTURES_SET)}
idx2gesture = {i: g for g, i in gesture2idx.items()}
num_classes = len(ALL_GESTURES_SET)
print("Num gesture classes:", num_classes)



Num gesture classes: 18


In [4]:

#print("Loaded scenarios:", len(splits_store))
#print(list(splits_store.keys())[:5])  # show a few keys

In [5]:
df_cebra_res=pd.read_csv('res/cebra_feats.csv')

In [6]:
df_cebra_res

,drop_thermal_and_tof_options,use_raw_imus,use_physics_features,num_features,cebra_output_dims,train_dist,val_dist,data_dist,cebra_model_path
0,True,False,True,24,2,"Counter({1: 4090, 0: 2430})","Counter({1: 1023, 0: 608})","Counter({1: 5113, 0: 3038})",models/cebra_dtt_1_uri_0_uff_1_ced_2.pt
1,True,True,False,11,2,"Counter({1: 4090, 0: 2430})","Counter({1: 1023, 0: 608})","Counter({1: 5113, 0: 3038})",models/cebra_dtt_1_uri_1_uff_0_ced_2.pt
2,True,True,True,31,2,"Counter({1: 4090, 0: 2430})","Counter({1: 1023, 0: 608})","Counter({1: 5113, 0: 3038})",models/cebra_dtt_1_uri_1_uff_1_ced_2.pt
3,False,False,False,332,2,"Counter({1: 4090, 0: 2430})","Counter({1: 1023, 0: 608})","Counter({1: 5113, 0: 3038})",models/cebra_dtt_0_uri_0_uff_0_ced_2.pt
4,False,False,True,349,2,"Counter({1: 4090, 0: 2430})","Counter({1: 1023, 0: 608})","Counter({1: 5113, 0: 3038})",models/cebra_dtt_0_uri_0_uff_1_ced_2.pt
5,False,True,False,336,2,"Counter({1: 4090, 0: 2430})","Counter({1: 1023, 0: 608})","Counter({1: 5113, 0: 3038})",models/cebra_dtt_0_uri_1_uff_0_ced_2.pt
6,False,True,True,356,2,"Counter({1: 4090, 0: 2430})","Counter({1: 1023, 0: 608})","Counter({1: 5113, 0: 3038})",models/cebra_dtt_0_uri_1_uff_1_ced_2.pt
7,True,False,True,24,8,"Counter({1: 4090, 0: 2430})","Counter({1: 1023, 0: 608})","Counter({1: 5113, 0: 3038})",models/cebra_dtt_1_uri_0_uff_1_ced_8.pt
8,True,True,False,11,8,"Counter({1: 4090, 0: 2430})","Counter({1: 1023, 0: 608})","Counter({1: 5113, 0: 3038})",models/cebra_dtt_1_uri_1_uff_0_ced_8.pt
9,True,True,True,31,8,"Counter({1: 4090, 0: 2430})","Counter({1: 1023, 0: 608})","Counter({1: 5113, 0: 3038})",models/cebra_dtt_1_uri_1_uff_1_ced_8.pt


In [7]:
drop_thermal_and_tof_options=[True,False]
use_raw_imus_options=[False,True]
use_physics_features_options=[False,True]
cebra_output_dims_list=[2,8,20,32]
num_kernels_list=[500,10000,5000,1000]

classifxn_res=[]

for cebra_output_dims in cebra_output_dims_list:
    for drop_thermal_and_tof in drop_thermal_and_tof_options:
        for use_raw_imus in use_raw_imus_options:
            for use_physics_features in use_physics_features_options:
                if drop_thermal_and_tof and not use_physics_features and not use_raw_imus:
                    continue
                scenario_key=f'dtt={int(drop_thermal_and_tof)}_uri={int(use_raw_imus)}_uff={int(use_physics_features)}_ced={int(cebra_output_dims)}'
                print(scenario_key)
                with open("res/all_splits.pkl", "rb") as f:                    
                    splits_store = pickle.load(f)

                if scenario_key not in splits_store:
                    del splits_store                
                    gc.collect()
                    continue
                X_train = splits_store[scenario_key]["X_train"]
                y_train = splits_store[scenario_key]["y_train"]
                X_val   = splits_store[scenario_key]["X_val"]
                y_val   = splits_store[scenario_key]["y_val"]
                del splits_store                
                gc.collect()

                print(X_train.shape, y_train.shape, X_val.shape, y_val.shape)
                X_train_flat = X_train.reshape(-1, X_train.shape[-1])
                X_val_flat   = X_val.reshape(-1, X_val.shape[-1])
                # read the model
                model_path=f'models/cebra_dtt_{int(drop_thermal_and_tof)}_uri_{int(use_raw_imus)}_uff_{int(use_physics_features)}_ced_{int(cebra_output_dims)}.pt'
                loaded_cebra_model = cebra.CEBRA.load(model_path)
                val_embedding_flat = loaded_cebra_model.transform(X_val_flat)
                train_embedding_flat = loaded_cebra_model.transform(X_train_flat)

                train_embedding = train_embedding_flat.reshape(
                    X_train.shape[0],   # 6520 trials
                    X_train.shape[1],   # 103 time steps
                    -1                  # latent dim (8)
                )

                val_embedding = val_embedding_flat.reshape(
                    X_val.shape[0],
                    X_val.shape[1],
                    -1
                )

                train_means = train_embedding.mean(axis=1)
                val_means = val_embedding.mean(axis=1)
                for num_kernels in num_kernels_list:

                    mr_results=minirocket_results(train_embedding,y_train,val_embedding,y_val,plot=False,
                                                  num_kernels=num_kernels)                

                    # prefix the metric keys
                    mr_prefixed = {f"mr_{k}": v for k, v in mr_results.items()}                

                    res_dic={
                        'drop_thermal_and_tof_options':drop_thermal_and_tof,
                        'use_raw_imus':use_raw_imus,
                        'use_physics_features':use_physics_features,
                        'num_features':X_train.shape[-1],
                        'cebra_output_dims':cebra_output_dims,
                        'train_dist':Counter(y_train),
                        'val_dist':Counter(y_val),   
                        'num_kernels':num_kernels,
                        **mr_results,
                        }
                    classifxn_res.append(res_dic)
                    classifxn_res_df=pd.DataFrame(classifxn_res)
                    classifxn_res_df.to_csv('res/mr_classifxn.csv',index=False)
                    

                

dtt=1_uri=0_uff=1_ced=2
(6520, 103, 24) (6520,) (1631, 103, 24) (1631,)
(6520, 2, 103)


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


              precision    recall  f1-score   support

           0       0.70      0.68      0.69       608
           1       0.81      0.83      0.82      1023

    accuracy                           0.77      1631
   macro avg       0.76      0.76      0.76      1631
weighted avg       0.77      0.77      0.77      1631

(6520, 2, 103)
              precision    recall  f1-score   support

           0       0.72      0.71      0.71       608
           1       0.83      0.84      0.83      1023

    accuracy                           0.79      1631
   macro avg       0.77      0.77      0.77      1631
weighted avg       0.79      0.79      0.79      1631

(6520, 2, 103)
              precision    recall  f1-score   support

           0       0.71      0.71      0.71       608
           1       0.83      0.83      0.83      1023

    accuracy                           0.78      1631
   macro avg       0.77      0.77      0.77      1631
weighted avg       0.78      0.78      0.78 

In [8]:
X_train.shape[1]

103